# Baseline Evaluation (mAP, F1)
Runs src/run_rfdetr_baseline_eval.py and prints output paths.

In [26]:
from pathlib import Path
import os
import sys
import site

def find_repo_root(marker='pyproject.toml', max_depth=10):
    """Search upward from current directory for repo marker file."""
    current = Path.cwd().resolve()
    for _ in range(max_depth):
        if (current / marker).exists():
            return current
        current = current.parent
    return None

def prefer_venv_site_packages():
    """Keep venv site-packages before cluster/global package paths."""
    venv_prefix = str(Path(sys.prefix).resolve())
    venv_sites = [p for p in site.getsitepackages() if p.startswith(venv_prefix)]
    cleaned = [p for p in sys.path if not (p.startswith('/net/software/') and 'site-packages' in p)]
    for p in reversed(venv_sites):
        if p in cleaned:
            cleaned.remove(p)
        cleaned.insert(0, p)
    sys.path[:] = cleaned

    # If regex was imported earlier from cluster paths, evict it so next import uses venv.
    cached = sys.modules.get('regex')
    cached_path = getattr(cached, '__file__', '') if cached is not None else ''
    if cached_path.startswith('/net/software/'):
        for name in list(sys.modules.keys()):
            if name == 'regex' or name.startswith('regex.'):
                del sys.modules[name]

REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    REPO_ROOT = Path('/net/tscratch/people/plgbmruszaj/dermatoscopy_ai')

RFD_XAI_ROOT = REPO_ROOT / 'dermato_ai' / 'rfdetr-xai'
os.chdir(RFD_XAI_ROOT)
sys.path.insert(0, str(RFD_XAI_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT))
prefer_venv_site_packages()

from dermato_ai.coco.dataset_manager import COCODatasetManager

# Data source: newest archive in repo data/coco_images_zip
DATA_ZIP_DIR = REPO_ROOT / 'data' / 'coco_images_zip'
manager = COCODatasetManager.from_archive(
    directory=DATA_ZIP_DIR,
    archive_pattern='*.zip',
)
dataset_dir = Path(manager.json_path).parent
dedup_coco_json = dataset_dir / 'result_dedup_for_eval.json'
manager.save(str(dedup_coco_json))
COCO_JSON = str(dedup_coco_json)
IMAGES_DIR = str(dataset_dir / 'images')

# Model from latest trained run
CHECKPOINT = '/net/tscratch/people/plgbmruszaj/rfdetr_all_data_results_yellow_large_20260507_150528/rfdetr_large_best_2573873/checkpoints/checkpoint_best_ema.pth'
MODEL_SIZE = 'large'
OUTPUT_DIR = str((RFD_XAI_ROOT / 'wyniki' / 'rfdetr_baseline').resolve())

print('✓ Python executable:', sys.executable)
print('✓ Working dir:', Path.cwd())
print('✓ Data zip dir:', DATA_ZIP_DIR)
print('✓ COCO JSON:', COCO_JSON)
print('✓ Images dir:', IMAGES_DIR)
print('✓ Checkpoint:', CHECKPOINT)
print('✓ Output dir:', OUTPUT_DIR)

INFO: Found newest archive: project-5-at-2026-05-07-11-56-9b7fa76c.zip
INFO: Unpacking project-5-at-2026-05-07-11-56-9b7fa76c.zip to /net/tscratch/people/plgbmruszaj/dermatoscopy_ai/data/coco_images_zip/extracted
INFO: Extraction complete
INFO: Found result.json at: /net/tscratch/people/plgbmruszaj/dermatoscopy_ai/data/coco_images_zip/extracted/result.json
INFO: Loading COCO dataset from /net/tscratch/people/plgbmruszaj/dermatoscopy_ai/data/coco_images_zip/extracted/result.json
INFO: Loaded 314 images, 1299 annotations
INFO: Collapsed duplicate image passes: removed 97 image entries and 0 annotations
INFO: Valid COCO structure: 217 images, 1299 annotations, 10 categories
INFO: Fixed 217 image paths
INFO: Saved dataset to /net/tscratch/people/plgbmruszaj/dermatoscopy_ai/data/coco_images_zip/extracted/result_dedup_for_eval.json


✓ Python executable: /net/tscratch/people/plgbmruszaj/dermatoscopy_ai/.venv/bin/python
✓ Working dir: /net/tscratch/people/plgbmruszaj/dermatoscopy_ai/dermato_ai/rfdetr-xai
✓ Data zip dir: /net/tscratch/people/plgbmruszaj/dermatoscopy_ai/data/coco_images_zip
✓ COCO JSON: /net/tscratch/people/plgbmruszaj/dermatoscopy_ai/data/coco_images_zip/extracted/result_dedup_for_eval.json
✓ Images dir: /net/tscratch/people/plgbmruszaj/dermatoscopy_ai/data/coco_images_zip/extracted/images
✓ Checkpoint: /net/tscratch/people/plgbmruszaj/rfdetr_all_data_results_yellow_large_20260507_150528/rfdetr_large_best_2573873/checkpoints/checkpoint_best_ema.pth
✓ Output dir: /net/tscratch/people/plgbmruszaj/dermatoscopy_ai/dermato_ai/rfdetr-xai/wyniki/rfdetr_baseline


In [27]:
import csv
import json
import logging
from collections import defaultdict
from pathlib import Path
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from rfdetr_eval_config import DetectionEvalConfig, coco_bbox_xywh_to_xyxy, greedy_match_single_class
from rfdetr_load import load_rfdetr

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

def _sorted_categories(coco: COCO):
    return sorted(coco.loadCats(coco.getCatIds()), key=lambda c: c["id"])

def _coco_predictions_for_images(model, coco: COCO, images_dir: Path, conf_thresh: float, idx_to_cat_id: list):
    preds = []
    next_id = 1
    img_ids = coco.getImgIds()
    for img_id in img_ids:
        info = coco.loadImgs(img_id)[0]
        path = images_dir / info["file_name"]
        if not path.is_file():
            alt = images_dir / "train" / info["file_name"]
            path = alt if alt.is_file() else path
        if not path.is_file():
            continue
        pil = Image.open(path).convert("RGB")
        det = model.predict(pil, threshold=conf_thresh)
        if det is None or len(det) == 0:
            continue
        for i in range(len(det)):
            x1, y1, x2, y2 = det.xyxy[i].tolist()
            w, h = x2 - x1, y2 - y1
            mid = int(det.class_id[i])
            if mid < 0 or mid >= len(idx_to_cat_id):
                continue
            coco_cat = int(idx_to_cat_id[mid])
            preds.append({
                "id": next_id,
                "image_id": int(img_id),
                "category_id": coco_cat,
                "bbox": [float(x1), float(y1), float(w), float(h)],
                "score": float(det.confidence[i]),
                "area": float(max(w, 0) * max(h, 0)),
                "iscrowd": 0,
            })
            next_id += 1
    return preds, next_id

def _accumulate_f1_matches(model, coco: COCO, images_dir: Path, cfg: DetectionEvalConfig, cat_id_to_idx: dict, num_classes: int):
    counts = {c: {"tp": 0, "fp": 0, "fn": 0} for c in range(num_classes)}

    for img_id in coco.getImgIds():
        info = coco.loadImgs(img_id)[0]
        path = images_dir / info["file_name"]
        if not path.is_file():
            alt = images_dir / "train" / info["file_name"]
            path = alt if alt.is_file() else path
        if not path.is_file():
            continue

        anns = coco.loadAnns(coco.getAnnIds(imgIds=[img_id]))
        pil = Image.open(path).convert("RGB")
        det = model.predict(pil, threshold=cfg.confidence_threshold)
        det_empty = det is None or len(det) == 0
        pred_boxes = [] if det_empty else det.xyxy.tolist()
        pred_scores = [] if det_empty else det.confidence.tolist()
        pred_cls = [] if det_empty else det.class_id.astype(int).tolist()

        gt_by_c = defaultdict(list)
        for ann in anns:
            if ann.get("iscrowd", 0):
                continue
            # Skip annotations for classes not in our evaluation set
            if ann["category_id"] not in cat_id_to_idx:
                continue
            cid = cat_id_to_idx[ann["category_id"]]
            gt_by_c[cid].append(coco_bbox_xywh_to_xyxy(ann["bbox"]))

        preds_by_c = defaultdict(list)
        if not det_empty:
            for i, cls in enumerate(pred_cls):
                if cls < 0 or cls >= num_classes:
                    continue
                preds_by_c[int(cls)].append((tuple(pred_boxes[i]), float(pred_scores[i])))

        for c in range(num_classes):
            gt_boxes = gt_by_c.get(c, [])
            plist = preds_by_c.get(c, [])
            if not plist:
                counts[c]["fn"] += len(gt_boxes)
                continue
            pred_boxes_c = [t[0] for t in plist]
            pred_scores_c = [t[1] for t in plist]
            matches = greedy_match_single_class(pred_boxes_c, pred_scores_c, gt_boxes, cfg.iou_match_threshold)
            tp = len(matches)
            fp = len(pred_boxes_c) - tp
            fn = len(gt_boxes) - tp
            counts[c]["tp"] += tp
            counts[c]["fp"] += fp
            counts[c]["fn"] += fn

    return counts

def _f1(tp: int, fp: int, fn: int, eps: float = 1e-12) -> float:
    prec = tp / max(tp + fp, eps)
    rec = tp / max(tp + fn, eps)
    return float(2 * prec * rec / max(prec + rec, eps))

# Run evaluation
cfg = DetectionEvalConfig(
    iou_match_threshold=0.5,
    confidence_threshold=0.25,
    run_label="full_dataset_eval",
)
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg.to_json(OUTPUT_DIR / "eval_config.json")

coco_json_path = Path(COCO_JSON)
coco = COCO(str(coco_json_path))
cats = _sorted_categories(coco)
cat_ids = [c["id"] for c in cats]
cat_id_to_idx = {cid: i for i, cid in enumerate(cat_ids)}
idx_to_cat_id = cat_ids
num_classes = len(cat_ids)

model = load_rfdetr('large', CHECKPOINT)
model_class_names = list(model.class_names) if model.class_names else [str(i) for i in range(num_classes)]

# Handle class mismatch: find COCO category that matches model's class name
if len(model_class_names) < num_classes:
    logger.info(f"Model has {len(model_class_names)} classes but COCO has {num_classes}")
    logger.info(f"Model classes: {model_class_names}")
    logger.info(f"COCO categories: {[(c['id'], c['name']) for c in cats]}")
    
    # Find which COCO category matches the model's class
    model_class_name = model_class_names[0]
    matching_cat = None
    for cat in cats:
        if model_class_name.lower() in cat['name'].lower() or cat['name'].lower() in model_class_name.lower():
            matching_cat = cat['id']
            logger.info(f"Found matching category: '{model_class_name}' -> COCO ID {matching_cat} ('{cat['name']}')")
            break
    
    if matching_cat is not None:
        # Remap: only evaluate the matching category
        class_names = model_class_names
        idx_to_cat_id = [matching_cat]
        cat_id_to_idx = {matching_cat: 0}
        num_classes = 1
        logger.info(f"Filtered to evaluate only category {matching_cat}")
    else:
        # No matching category found - use first category as fallback
        logger.warning(f"No matching category found for '{model_class_name}' in COCO data - using category 0 as fallback")
        class_names = model_class_names
        idx_to_cat_id = [cat_ids[0]]
        cat_id_to_idx = {cat_ids[0]: 0}
        num_classes = 1
else:
    class_names = model_class_names

preds, _ = _coco_predictions_for_images(model, coco, Path(IMAGES_DIR), cfg.confidence_threshold, idx_to_cat_id)

coco_gt = coco
coco_dt = coco_gt.loadRes(preds)
ev = COCOeval(coco_gt, coco_dt, "bbox")
# Restrict evaluation to only the categories the model was trained on,
# so mAP is not averaged over other categories with AP=0.
ev.params.catIds = idx_to_cat_id
# Also restrict to images that have GT annotations for these categories,
# matching the training evaluation protocol.
ev.params.imgIds = coco.getImgIds(catIds=idx_to_cat_id)
ev.evaluate()
ev.accumulate()
ev.summarize()

map_5095 = float(ev.stats[0])
map_50 = float(ev.stats[1])

f1_counts = _accumulate_f1_matches(model, coco, Path(IMAGES_DIR), cfg, cat_id_to_idx, num_classes)
f1_per_class = {class_names[i]: _f1(**f1_counts[i]) for i in range(num_classes)}

summary = {
    "protocol": "full_dataset_eval",
    "checkpoint": str(Path(CHECKPOINT).resolve()),
    "coco_json": str(coco_json_path.resolve()),
    "images_dir": str(Path(IMAGES_DIR).resolve()),
    "mAP_50_95": map_5095,
    "mAP_50": map_50,
    "f1_per_class": f1_per_class,
    "class_names": class_names,
}
(OUTPUT_DIR / "baseline_metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

csv_path = OUTPUT_DIR / "baseline_metrics_per_class.csv"
with csv_path.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["class_name", "f1"])
    for name, sc in f1_per_class.items():
        w.writerow([name, f"{sc:.6f}"])
    w.writerow(["__mAP_50__", f"{map_50:.6f}"])
    w.writerow(["__mAP_50_95__", f"{map_5095:.6f}"])

print(json.dumps({"wrote": str(OUTPUT_DIR), "mAP_50": map_50, "mAP_50_95": map_5095}, indent=2))

[2026-05-07 18:06:08] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-07 18:06:08] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


[2026-05-07 18:06:09] [WARNING] rf-detr - Checkpoint has 1 classes but model is configured for 90. Using checkpoint class count (1). Pass num_classes=1 to suppress this warning.
INFO: Model has 1 classes but COCO has 10
INFO: Model classes: ['Yellow globlues (ulcer)']
INFO: COCO categories: [(0, 'Blood vessels'), (1, 'Blue-gray globules'), (2, 'Large blue gray ovoid nests'), (3, 'Linear segmented structures (maple leaf-like areas)'), (4, 'MAY globules'), (5, 'Milia-like-cyst'), (6, 'Rosettes'), (7, 'White circle'), (8, 'White lines'), (9, 'Yellow globlues (ulcer)')]
INFO: Found matching category: 'Yellow globlues (ulcer)' -> COCO ID 9 ('Yellow globlues (ulcer)')
INFO: Filtered to evaluate only category 9
[2026-05-07 18:06:09] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().


Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.04s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.646
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.955
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.736
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.306
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.611
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.738
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.336
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.679
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.679
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10